# 54. Capstone: an evidence assistant with regression tests

**Level:** Advanced

**Prerequisites:** 41-43 and 52: schemas, retrieval, evaluation, bounded tools.

[Course index](00_START_HERE.ipynb) | [Previous](53_Capstone_Network_Telemetry_Classifier.ipynb) | [Next](55_Recent_Advances_and_Reading_Research.ipynb)

**Topics:** Capstone, RAG, Structured output, Abstention, Regression tests, Evidence.

**Matching script:** `examples\54_Capstone_Evidence_Assistant_and_Evals.py`

**How to study:** read the notes, predict an example's output, run it, trace each change in data/shape/state, then complete the practice task. Every worked code cell contains its own imports and data and can run independently of other notebooks.

**Execution contract:** Python 3.11+, course dependencies, CPU only, synthetic/local data. No hosted API requests, model-weight downloads, dataset downloads, listening web servers, credentials, or real device changes. The temporary-file lesson cleans up its own files. Expected errors are caught narrowly and asserted; unexpected errors must remain visible.

## Concept notes and mental model

This capstone joins strict output schemas, retrieval, source IDs, and a small evaluation suite.
The answering component is deliberately extractive: it returns the matched policy sentence
instead of calling a hidden hosted model. This makes provenance and failure cases inspectable.

A future LLM generator must be tested separately for factual support, citations, refusals,
prompt-injection handling, privacy, latency, and cost. Keep the same schema and held-out
cases when swapping implementations so improvements and regressions are measurable.

Document access control must precede retrieval disclosure. A query with no relevant evidence
should yield unknown, not an invented policy. An exact string evaluation is appropriate here
because the output is intentionally a constrained extract; free-form paraphrases need richer
scoring and human review.

## Mathematics or data contract

Evaluate retrieval support, output-schema validity, and answer correctness as separate invariants.

## Important limitation

Lexical matching on three local documents is not general semantic retrieval or a deployed LLM assistant.

Distinguish what is implemented and asserted below from what would need additional data, training, independent evaluation, authorization, or operational controls.

## Worked example 1: Build and test a local evidence contract

**Purpose:** Return only supported extracts or explicit unknown.

### Read the code in this order
1. Define a validated output record.
2. Index a small synthetic policy set.
3. Use a deterministic overlap threshold and source IDs.
4. Run known and unsupported evaluation cases.

### Syntax and behavior to notice
- `transform(...)` applies already-fitted preprocessing; it does not refit it.
- `assert` checks the stated behavior; a failed check raises AssertionError.

### Expected output
This is the output captured from executing the worked example in the recorded environment. Package versions and floating-point rounding can change numeric details; the assertions express the required shapes, invariants, and thresholds.

```text
passed evidence cases: 3
{'text': 'Lab interface MTU is 1500 bytes.', 'source_ids': ['mtu']}
```

**Before running:** identify the input variables, which values are learned versus chosen, and what each assertion proves. A successful shape test does not by itself prove predictive quality. Training-only loss checks are labeled as such.

In [1]:
import numpy as np
from pydantic import BaseModel, ConfigDict
from sklearn.feature_extraction.text import TfidfVectorizer

class Answer(BaseModel):
    model_config = ConfigDict(extra="forbid", strict=True)
    text: str
    source_ids: list[str]

docs = {"mtu": "Lab interface MTU is 1500 bytes.",
        "backup": "Configuration backup frequency is daily.",
        "dns": "Lab DNS resolver is 192.0.2.53."}
ids = list(docs)
vectorizer = TfidfVectorizer()
matrix = vectorizer.fit_transform(docs.values())
def answer(query):
    scores = (matrix @ vectorizer.transform([query]).T).toarray().ravel()
    best = int(np.argmax(scores))
    if scores[best] < 0.15:
        return Answer(text="unknown", source_ids=[])
    return Answer(text=docs[ids[best]], source_ids=[ids[best]])
tests = [("MTU interface bytes", "mtu"), ("backup frequency", "backup"),
         ("penguin habitat", None)]
for query, expected_id in tests:
    result = answer(query)
    assert result.source_ids == ([] if expected_id is None else [expected_id])
    assert result.text == ("unknown" if expected_id is None else docs[expected_id])
print("passed evidence cases:", len(tests))
print(answer("MTU interface bytes").model_dump())

passed evidence cases: 3
{'text': 'Lab interface MTU is 1500 bytes.', 'source_ids': ['mtu']}


### Interpret example 1

The result should demonstrate this purpose: **Return only supported extracts or explicit unknown.**

Follow the assertions back to the exact data or state being checked. Where a model was fitted, distinguish training fit from held-out evaluation. Where input is rejected, inspect whether rejection happens before any intended mutation or tool execution. Do not delete a failed assertion to make an incorrect result look valid.

## Practice: extend the concept yourself

Add stale and conflicting documents and require an explicit ambiguity outcome rather than arbitrary selection.

1. Write your expected result before editing code.
2. Add one normal-case test and one boundary/failure-case test.
3. Keep training/validation/test responsibilities separate where relevant.
4. Report an honest failure or unknown result rather than a success-shaped fallback.

The task is intentionally not filled in for you; the worked examples above are complete and runnable. Add your own code cell for the extension.

## Understanding checkpoint
- Explain `Capstone` in your own words and give one example of when it is useful.
- Trace one worked example from input through transformation to returned or printed output.
- State this lesson's mathematical/data contract: Evaluate retrieval support, output-schema validity, and answer correctness as separate invariants.
- Name the boundary of the demonstration: Lexical matching on three local documents is not general semantic retrieval or a deployed LLM assistant.
- Describe one test that could catch a plausible but wrong implementation.

## Primary references and further reading

- `https://scikit-learn.org/stable/user_guide.html`
- `https://docs.pydantic.dev/latest/concepts/models/`

Research snapshot: **2026-10-17**. Foundational references need not be recent. Mutable library docs describe their current version; compare with the tested dependency versions in the course manifest. References are reading material, not runtime downloads.